### Setup

In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from common.utils import DataPreprocessor, FeatureEngineer, set_seed
from common.exp_data_utils import ExperimentDataPreprocessor
from common.eval import Evaluator

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 42

# Initialize data processors
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()
feature_engineer = FeatureEngineer()
experiment_data_preprocessor = ExperimentDataPreprocessor()
evaluator = Evaluator()


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 42
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42
torch set to use deterministic algorithms


### Load and Process DataFrame

In [2]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
    rating_threshold=4.0, # replica 3.5 rating threshold
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (10/0):
Data count before: 480608
Data count after: 480448
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480448
Num of distinct users: 2064
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Join Side Information

In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5, threshold=5
)
interaction_info_df.head()

extracting item features...
merging features...
interaction data count before merging: 480448
interaction data count after merging: 478404
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[[RARE], [RARE], [RARE], [RARE], [RARE]]",USA,[RARE],Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,[RARE],Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[[RARE], laura_linney, ernie_hudson_jr, tim_cu...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


### Prepare Train/Valid/Test Set

In [4]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.75, # replica 64% for training
    val_ratio=0.10, # replica 16% for validation
    test_ratio=0.15, # replica 20% for testing
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.75 : 0.1 : 0.15):
train: 358027 (74.84%)
valid: 46916 (9.81%)
test: 73461 (15.36%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.555944
1    0.444056
Name: proportion, dtype: float64
valid label
0    0.610772
1    0.389228
Name: proportion, dtype: float64
test label
0    0.585277
1    0.414723
Name: proportion, dtype: float64


### Re-index User/Item ID & Encode Categorical Features

In [5]:
print("Train: fit_transform")
encoded_train_df = feature_engineer.fit_transform(train_df)
print("---"*10)
print("Valid: transform")
encoded_valid_df = feature_engineer.transform(valid_df)
print("---"*10)
print("Test: transform")
encoded_test_df = feature_engineer.transform(test_df)
print("---"*10)

Train: fit_transform
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
Fitted: user/item mapping
Fitted: vocab2idx for actorID
Fitted: vocab2idx for country
Fitted: vocab2idx for directorID
Fitted: vocab2idx for genre
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Valid: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Test: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------


In [6]:
# # NOTE: can check the encoding vocab idx content from the feature engineer
# oov_idx = feature_engineer.vocab2idx["movieID"]["[OOV]"]
# len(test_df[test_df["movieID"] == oov_idx])

### Prepare Additional Data for Train/Inference

#### Build bi-partite graph for training

In [7]:
# NOTE: At training, we use interaction graph from train_df for train and validation
train_graph = experiment_data_preprocessor.create_interaction_graph(encoded_train_df)

# NOTE: At inference, we can use graph of (train_df + valid_df)
# train_valid_graph = utils.create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

Creating interaction graph...
Drop negative samples
  Num of all interactions: 358027
  Num of positive interactions: 158984 

Building edges...
Building labels...
Interaction Graph: Data(edge_index=[2, 317967], edge_label=[158984])
Edge Index: tensor([[    0,     0,     0,  ..., 10758, 10762, 10763],
        [ 2089,  2202,  2318,  ...,  1760,  1645,  1760]])


#### Evaluate User Diversity Preference Scale

In [8]:
user_dps_df = evaluator.eval_user_diversity_preference_scale(encoded_train_df, feature_engineer.vocab2idx, normalized=True, rescale=True)
user_dps_df.head(1)


Calculating user diversity preference scale:   0%|          | 0/2064 [00:00<?, ?it/s]

Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:06<00:00, 343.05it/s]


,userID,actorID_wvec,actorID_dps,country_wvec,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps
0,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 0.0, 0.0, ...",0.370445,"[0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 0.0, 0.0, 0.0, ...",0.292809,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.480054,"[97.0, 41.5, 8.0, 5.0, 39.0, 45.5, 0.0, 44.5, ...",0.645735


In [9]:
user_dps_df.describe()

,userID,actorID_dps,country_dps,directorID_dps,genre_dps
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.600991,0.409910,0.607440,0.775214
std,595.969798,0.190923,0.155270,0.165699,0.111637
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,515.750000,0.474224,0.305642,0.491126,0.730144
50%,1031.500000,0.631066,0.397638,0.625725,0.801460
75%,1547.250000,0.747282,0.505837,0.732749,0.851511
max,2063.000000,1.000000,1.000000,1.000000,1.000000


#### Prepare Item Multihot Vec on Each Dimension

In [10]:
item_vec_df = evaluator.get_item_feature_multihot_vec(encoded_train_df, feature_engineer.vocab2idx)
item_vec_df.head(1)

,movieID,actorID_vec,country_vec,directorID_vec,genre_vec
0,1588,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


#### Prepare train/valid triplet data

In [11]:
train_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_train_df, k_negative_samples=5)
train_triplet_with_dps_df = train_triplet_df.merge(user_dps_df, on="userID", how="left")
train_triplet_with_dps_df = train_triplet_with_dps_df.merge(item_vec_df, left_on="pos_item_id", right_on="movieID", how="left")
train_triplet_with_dps_df.head(1)

# valid_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_valid_df, k_negative_samples=5)
# valid_triplet_with_dps_df = valid_triplet_df.merge(user_dps_df, on="userID", how="left")
# valid_triplet_with_dps_df = valid_triplet_with_dps_df.merge(item_vec_df, left_on="pos_item_id", right_on="movieID", how="inner")
# valid_triplet_with_dps_df.head(1)

Original data count (positive samples): 158984
Num of triplets: 158984(pos samples) * 5(negative sampled items) = 794920


,userID,pos_item_id,neg_item_id,actorID_idx,country_idx,directorID_idx,genre_idx,neg_actorID_idx,neg_country_idx,neg_directorID_idx,...,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps,movieID,actorID_vec,country_vec,directorID_vec,genre_vec
0,0,1102,307,"[2267, 1401, 582, 998, 1847]",37,360,"[2, 3, 17, 18, 0, 0, 0, 0]","[1769, 713, 931, 1356, 1]",12,452,...,0.292809,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.480054,"[97.0, 41.5, 8.0, 5.0, 39.0, 45.5, 0.0, 44.5, ...",0.645735,1102,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


#### Prepare prediction pool for inference/testing

In [12]:
# NOTE: Prepare prediction pool to evaluate the model
validation_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_valid_df, K=1000)
prediction_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_test_df, K=1000)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2063
Item Pool: 6098, negative sampled to 1000 items for each user
Num of interactions: 2063(users) * 1000(items) = 2063000
Prediction DataFrame:
User Pool: 2064
Item Pool: 6959, negative sampled to 1000 items for each user
Num of interactions: 2064(users) * 1000(items) = 2064000


,userID,movieID,label,actorID_idx,country_idx,directorID_idx,genre_idx
2063995,2063,1660,0,"[1, 735, 1165, 1, 1]",37,1,"[4, 5, 0, 0, 0, 0, 0, 0]"
2063996,2063,2347,0,"[1, 1, 1, 1, 1]",18,134,"[4, 5, 0, 0, 0, 0, 0, 0]"
2063997,2063,4479,0,"[867, 884, 416, 2282, 1]",37,373,"[6, 17, 0, 0, 0, 0, 0, 0]"
2063998,2063,5256,0,"[865, 1756, 1810, 1, 1]",37,358,"[9, 0, 0, 0, 0, 0, 0, 0]"
2063999,2063,78,0,"[1, 1356, 1, 1, 1]",37,1,"[9, 16, 0, 0, 0, 0, 0, 0]"


### Prepare DataLoader

In [13]:
# NOTE: ensure reproducibility of DataLoader
import torch
from common.utils import seed_worker
g = torch.Generator()
g.manual_seed(RANDOM_SEED)

# TODO: determine which Dataset to use
from torch.utils.data import DataLoader
from common.datasets import TripletDataset, UserItemPairDataset, UserPosItemSampler, get_user_triplet_mapping

BATCH_SIZE = 1024

train_dataset = TripletDataset(train_triplet_with_dps_df)
# valid_dataset = TripletDataset(valid_triplet_with_dps_df)
valid_dataset = UserItemPairDataset(validation_pool_df)
test_dataset = UserItemPairDataset(prediction_pool_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))
print("test data count:", len(test_dataset))

# # TODO: Custom sampler
# MIN_POS_ITEMS = 2
# user_to_pos_items, user_pos_to_indices = get_user_triplet_mapping(train_triplet_with_dps_df, MIN_POS_ITEMS)
# train_sampler = UserPosItemSampler(user_to_pos_items, user_pos_to_indices, batch_size=BATCH_SIZE, min_pos_items=MIN_POS_ITEMS, max_pos_items=20, buffer=0)
# train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=4, worker_init_fn=seed_worker, generator=g)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, worker_init_fn=seed_worker, generator=g, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


train data count: 794920
valid data count: 2063000
test data count: 2064000


### Load Embeddings

In [14]:
model_name = "(mt)ngcf_v2_k5"
embeddings_dir = "embeddings/"
user_emb = torch.load(f"{embeddings_dir}{model_name}_user_emb.pt")
item_emb = torch.load(f"{embeddings_dir}{model_name}_item_emb.pt")

print(user_emb.size())
print(item_emb.size())


torch.Size([2065, 256])
torch.Size([8707, 256])


### Configure Model (LightningModule)

In [15]:
from lightning_models.finetune import FTDPRec

LR = 1e-3
EPOCHS = 50
REG_WEIGHT = 1e-5

DPS_WEIGHTS = {
    "actor_dps": 0.25,
    "country_dps": 0.25,
    "director_dps": 0.25,
    "genre_dps": 0.25,
}

DPR_WEIGHTS = {
    "actor_dpr": 0.25,
    "country_dpr": 0.25,
    "director_dpr": 0.25,
    "genre_dpr": 0.25,
}

DPM_WEIGHTS = {
    "actor_pd": 0.25,
    "country_pd": 0.25,
    "director_pd": 0.25,
    "genre_pd": 0.25,
}

MT_WEIGHTS = {
    # "bpr_loss": 1.0,
    "dps_loss": 1.0,
    "dpr_loss": 1.0,
    "dpm_loss": 1.0,
    "l2_loss": 1e-3,
}

REL_EMB_DIM = None  # Dimension of relation embeddings, can be adjusted based on model complexity
STRATEGY = "delta"  # "direct", "delta", "lora"
LORA_R = 2  # Rank for LoRA, can be adjusted based on model complexity
RESCALE_METHOD = None  # None, "log", "ema"

model = FTDPRec(
    user_emb_tensor=user_emb,
    item_emb_tensor=item_emb,
    strategy=STRATEGY,
    lora_r=LORA_R,
    rel_dim=REL_EMB_DIM,
    lr=LR,
    reg_weight=REG_WEIGHT,
    dps_weights=DPS_WEIGHTS,
    dpr_weights=DPR_WEIGHTS,
    dpm_weights=DPM_WEIGHTS,
    mt_weights=MT_WEIGHTS,
    rescale_method=RESCALE_METHOD,
)


Seed set to 42


### Configure Trainer and Experiment

In [16]:
from common._mlflow import get_mlflow_logger, get_callbacks

EXPERIMENT_NAME = "finetune-exp"
VERSION = "s2-v3"
RUN_NAME = "min-max_&_add-l2loss(1e-3)"
PATIENCE = 5
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME, tags={"version": VERSION})
trainer_callbacks = get_callbacks(
    exp_name=EXPERIMENT_NAME,
    version_name=VERSION,
    run_name=RUN_NAME,
    patience=PATIENCE,
    monitor_metric="val_ndcg20",
    monitor_mode="max",
    hyper_param_str=f"base-model-{model_name}-lr={LR}",
)

In [17]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=50,
    callbacks=trainer_callbacks,
    accelerator='auto',  # or 'auto', 'gpu'
    # devices=[0], # if gpu is available
)


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


### Train Model

In [18]:
# Start training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /home/adam/R11_Bai/DPRecSys/experiments/test_checkpoints/finetune-exp exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

   | Name                | Type              | Params | Mode 
-------------------------------------------------------------------
0  | embedding_model     | EmbeddingWrapper  | 5.5 M  | train
1  | bpr_loss            | BPRLoss           | 0      | train
2  | reg_loss            | EmbLoss           | 0      | train
3  | dps_module          | DPSPredictor      | 1.0 K  | train
4  | dps_loss_fn         | DPSLoss           | 0      | train
5  | dpr_module          | DPRegularizer     | 263 K  | train
6  | dpr_loss_fn         | DPRLoss           | 0      | train
7  | dpm_module          | DPMatcher         | 0      | train
8  | dpm_loss_fn         | KLDivergenceLoss  | 0      | train
9  | l2_regularizer      | L2Loss  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 4, global step 3885: 'val_ndcg20' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_ndcg20 did not improve in the last 5 records. Best score: 0.275. Signaling Trainer to stop.
Epoch 5, global step 4662: 'val_ndcg20' was not in top 1


🏃 View run min-max_&_add-l2loss(1e-3) at: http://140.112.106.216:3683/#/experiments/17/runs/4f9c638adaee47aea917fb0d787c713f
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/17


In [19]:
model.user_emb

tensor([[ 0.1091, -0.3534, -0.2669,  ..., -0.3638, -0.3496,  0.1736],
        [ 0.4092,  0.0795, -0.3786,  ...,  0.3696, -0.4366,  0.3803],
        [-0.0836,  0.0525, -0.3540,  ...,  0.3607, -0.1894,  0.3149],
        ...,
        [-0.2307, -0.0739,  0.1773,  ...,  0.0178, -0.3314, -0.0798],
        [-0.5329,  0.0702, -0.1687,  ...,  0.2265, -0.4596, -0.2240],
        [-0.0512,  0.0134,  0.0163,  ..., -0.0911, -0.1541, -0.0184]],
       device='cuda:0')

### Inference

In [20]:
# NOTE: the inference model MUST be the same as the training model
# best_model_experiment_name = "mtdp-ngcf-v2-exp"
# best_model_checkpoint_path = ""
# best_model_path = f"test_checkpoints/{best_model_experiment_name}/{best_model_checkpoint_path}"
best_model_path = trainer.checkpoint_callback.best_model_path

model = FTDPRec.load_from_checkpoint(checkpoint_path=best_model_path)
# start inference
trainer.test(model=model, dataloaders=test_loader)


Seed set to 42
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        test_ndcg10        │     0.329416960477829     │
│        test_ndcg20        │    0.3639160096645355     │
│        test_ndcg5         │    0.27536189556121826    │
│     test_precision10      │    0.11337209492921829    │
│     test_precision20      │    0.09861918538808823    │
│      test_precision5      │    0.12257751822471619    │
│       test_recall10       │     0.108480304479599     │
│       test_recall20       │    0.1766068935394287     │
│       test_recall5        │    0.06219065561890602    │
└───────────────────────────┴───────────────────────────┘

🏃 View run min-max_&_add-l2loss(1e-3) at: http://140.112.106.216:3683/#/experiments/17/runs/4f9c638adaee47aea917fb0d787c713f
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/17


[{'test_ndcg5': 0.27536189556121826,
  'test_ndcg10': 0.329416960477829,
  'test_ndcg20': 0.3639160096645355,
  'test_precision5': 0.12257751822471619,
  'test_precision10': 0.11337209492921829,
  'test_precision20': 0.09861918538808823,
  'test_recall5': 0.06219065561890602,
  'test_recall10': 0.108480304479599,
  'test_recall20': 0.1766068935394287}]

In [21]:
model.test_results["eval_score_df"].describe()

,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.275362,0.062191,0.122578,0.329417,0.108480,0.113372,0.363916,0.176607,0.098619
std,595.969798,0.355570,0.120004,0.172557,0.320978,0.156125,0.129638,0.277182,0.194811,0.096832
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,515.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1031.500000,0.000000,0.000000,0.000000,0.333333,0.054054,0.100000,0.368280,0.125000,0.050000
75%,1547.250000,0.501266,0.083333,0.200000,0.543771,0.156661,0.200000,0.545324,0.250000,0.150000
max,2063.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.700000,1.000000,1.000000,0.600000


In [22]:
# Evaluate user diversity preference matching score (DPMS) at k
eval_df = evaluator.prepare_evaluation_data(model.test_results, feature_engineer.idx2vocab)

# Get user DPMS
user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

user_dpms_df.describe()

candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:02<00:00, 695.87it/s]

combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.176918,0.979758,0.190145,0.842438,0.547315
std,595.969798,0.084208,0.036115,0.124100,0.090720,0.061025
min,0.000000,0.000000,0.353897,0.000000,0.318945,0.276616
25%,515.750000,0.114936,0.978455,0.096400,0.799313,0.508807
50%,1031.500000,0.180043,0.988751,0.180202,0.865733,0.551344
75%,1547.250000,0.236071,0.994300,0.277181,0.906071,0.589798
max,2063.000000,0.436637,1.000000,0.666607,0.983217,0.723416


In [23]:
model.user_emb

tensor([[ 0.0961, -0.2113, -0.2398,  ..., -0.1701, -0.2681,  0.0668],
        [ 0.1445, -0.0291, -0.1171,  ...,  0.1525, -0.3073,  0.1534],
        [-0.0196,  0.0622, -0.0402,  ...,  0.2460, -0.0823,  0.0889],
        ...,
        [-0.2121,  0.0605,  0.1776,  ..., -0.1172, -0.2431, -0.1242],
        [-0.2765,  0.1064,  0.0914,  ..., -0.0866, -0.3449, -0.0297],
        [-0.0512,  0.0134,  0.0163,  ..., -0.0911, -0.1541, -0.0184]],
       device='cuda:0')